In [7]:
import json
import csv
import gzip
import os
import ast

datamap_file = "../data/toys/datamaps.json"
meta_file = "../raw/toys/meta_Toys_and_Games.json.gz"
output_file = "../data/toys/generation/item_feature/step1_positive/meta-toys.csv"

with open(datamap_file, "r", encoding="utf-8") as f:
    datamap = json.load(f)

id2item = datamap["id2item"]

asin2itemID = {asin: int(itemID) - 1 for itemID, asin in id2item.items()}
target_asins = set(asin2itemID.keys())

results = {}

with gzip.open(meta_file, "rt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            obj = ast.literal_eval(line)

        asin = obj.get("asin")
        if asin in target_asins:
            itemID = asin2itemID[asin]
            results[itemID] = {
                "itemID": itemID,
                "asin": asin,
                "title": obj.get("title", ""),
                "price": obj.get("price", ""),
                "imUrl": obj.get("imUrl", ""),
                "related": json.dumps(obj.get("related", {}), ensure_ascii=False),
                "brand": obj.get("brand", ""),
                "categories": json.dumps(obj.get("categories", []), ensure_ascii=False),
                "salesRank": json.dumps(obj.get("salesRank", {}), ensure_ascii=False),
                "description": obj.get("description", "")
            }

sorted_rows = [results[k] for k in sorted(results.keys())]

fieldnames = [
    "itemID", "asin", "title", "price", "imUrl",
    "related", "brand", "categories", "salesRank", "description"
]

os.makedirs(os.path.dirname(output_file), exist_ok=True)

with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(sorted_rows)

print(f"saved {output_file}")

saved ./generation/item_feature/step1_positive/meta-toys.csv
